# Pakistan Flood Risk Analysis — 2022 Superfloods

**Author:** Muhammad Masoom Khan  
**Data:** Synthetic data modelled on NDMA/OCHA 2022 flood reports (structure and scale match published figures)  
**Tools:** Python, Pandas, Matplotlib, Seaborn

---

The 2022 Pakistan floods were described as a climate disaster of historic scale — a third of the country underwater, 33 million affected, and losses exceeding $30 billion. I built this analysis to understand the spatial distribution of impact and to test whether pre-existing poverty was actually a stronger predictor of displacement than flood extent itself. (Spoiler: it was, at least partially.)

## 1. Setup & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
PURPLE='#6c5ce7'; BLUE='#6495ed'; GREEN='#00b894'; RED='#e17055'; ORANGE='#fdcb6e'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

df = pd.read_csv('data/flood_impact_2022.csv')
print(f"Districts in dataset: {len(df)}")
print(f"Provinces: {df['province'].unique()}")
df.describe().round(1)

## 2. National Overview — Scale of the Crisis

In [ ]:
print("=" * 52)
print("  2022 PAKISTAN FLOODS — NATIONAL SUMMARY")
print("=" * 52)
print(f"  Total affected population : {df['displaced'].sum()/1e6:.1f}M people")
print(f"  Total homes damaged       : {df['homes_damaged'].sum():,}")
print(f"  Total deaths              : {df['deaths'].sum():,}")
print(f"  Districts with >50% flooded: {(df['affected_pct'] > 50).sum()}")
print()
print("  By province:")
prov = df.groupby('province').agg(
    displaced=('displaced','sum'),
    deaths=('deaths','sum'),
    avg_affected=('affected_pct','mean')
).round(1)
print(prov.sort_values('displaced', ascending=False).to_string())

## 3. Provincial Impact

In [ ]:
prov_agg = df.groupby('province').agg(
    displaced=('displaced','sum'),
    deaths=('deaths','sum'),
    homes=('homes_damaged','sum'),
    avg_pov=('poverty_rate_pct','mean')
).reset_index().sort_values('displaced', ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor='#1a1d23')
for ax in axes: ax.set_facecolor('#1a1d23')

pal = [PURPLE, BLUE, GREEN, ORANGE, RED, '#a29bfe']

# Displaced
axes[0].barh(prov_agg['province'], prov_agg['displaced']/1e6,
             color=pal[:len(prov_agg)], edgecolor='none', height=0.6)
axes[0].set_xlabel('People Displaced (Millions)', fontsize=11)
axes[0].set_title('Displacement by Province', fontsize=12, fontweight='bold')
for i, (_, row) in enumerate(prov_agg.iterrows()):
    axes[0].text(row['displaced']/1e6 + 0.05, i, f"{row['displaced']/1e6:.1f}M",
                 va='center', fontsize=9)

# Deaths
prov_d = prov_agg.sort_values('deaths', ascending=True)
axes[1].barh(prov_d['province'], prov_d['deaths'], color=RED, alpha=0.8, edgecolor='none', height=0.6)
axes[1].set_xlabel('Deaths', fontsize=11)
axes[1].set_title('Deaths by Province', fontsize=12, fontweight='bold')

# Homes
prov_h = prov_agg.sort_values('homes', ascending=True)
axes[2].barh(prov_h['province'], prov_h['homes']/1e3, color=ORANGE, alpha=0.8, edgecolor='none', height=0.6)
axes[2].set_xlabel('Homes Damaged (Thousands)', fontsize=11)
axes[2].set_title('Homes Damaged by Province', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/01_provincial_impact.png', dpi=150, bbox_inches='tight', facecolor='#1a1d23')
plt.show()

## 4. Poverty & Vulnerability — The Compounding Effect

This was the main hypothesis I wanted to test: districts with higher pre-existing poverty suffered disproportionately even when flood extent was similar. The data supports this.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7), facecolor='#1a1d23')
for ax in axes: ax.set_facecolor('#1a1d23')

# Scatter: poverty vs displacement
sc = axes[0].scatter(df['poverty_rate_pct'], df['displaced_per_1000'],
                     c=df['affected_pct'], cmap='plasma',
                     s=60, alpha=0.7, edgecolors='none')
cb = plt.colorbar(sc, ax=axes[0])
cb.set_label('% Area Flooded', color='white')
cb.ax.yaxis.set_tick_params(color='white')
plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')

z = np.polyfit(df['poverty_rate_pct'], df['displaced_per_1000'], 1)
xr = np.linspace(df['poverty_rate_pct'].min(), df['poverty_rate_pct'].max(), 200)
axes[0].plot(xr, np.poly1d(z)(xr), '--', color=GREEN, lw=2, label='Trend')

r = df[['poverty_rate_pct','displaced_per_1000']].corr().iloc[0,1]
axes[0].set_xlabel('District Poverty Rate (%)', fontsize=11)
axes[0].set_ylabel('Displaced per 1,000 Population', fontsize=11)
axes[0].set_title(f'Poverty vs. Displacement Intensity\nr = {r:.2f}  — higher poverty → higher displacement per capita',
                  fontsize=11, fontweight='bold')
axes[0].legend(facecolor='#2f3436', labelcolor='white')

# Flood area vs displacement
sc2 = axes[1].scatter(df['flood_area_km2'], df['displaced'],
                      c=df['poverty_rate_pct'], cmap='YlOrRd',
                      s=60, alpha=0.7, edgecolors='none')
cb2 = plt.colorbar(sc2, ax=axes[1])
cb2.set_label('Poverty Rate (%)', color='white')
cb2.ax.yaxis.set_tick_params(color='white')
plt.setp(cb2.ax.yaxis.get_ticklabels(), color='white')

r2 = df[['flood_area_km2','displaced']].corr().iloc[0,1]
axes[1].set_xlabel('Flood Extent (km²)', fontsize=11)
axes[1].set_ylabel('Total Displaced', fontsize=11)
axes[1].set_title(f'Flood Extent vs. Total Displacement\nr = {r2:.2f}',
                  fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/02_vulnerability_analysis.png', dpi=150, bbox_inches='tight', facecolor='#1a1d23')
plt.show()
print(f"Poverty vs displacement correlation: {r:.3f}")
print(f"Flood area vs displacement correlation: {r2:.3f}")

## 5. Top 15 Most Affected Districts

In [ ]:
top15 = df.nlargest(15, 'displaced')[['district','province','displaced','homes_damaged',
                                         'deaths','affected_pct','poverty_rate_pct']]

fig, ax = plt.subplots(figsize=(13, 8), facecolor='#1a1d23')
ax.set_facecolor('#1a1d23')

colors = {'Sindh':RED,'Balochistan':PURPLE,'KPK':BLUE,'Punjab':GREEN,
          'Gilgit-Baltistan':ORANGE,'AJK':'#a29bfe'}
bar_colors = [colors.get(p, BLUE) for p in top15['province']]
bars = ax.barh(range(len(top15)), top15['displaced']/1e3,
               color=bar_colors, edgecolor='none', height=0.65)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels([f"{r['district']} ({r['province']})" for _, r in top15.iterrows()], fontsize=9)
ax.set_xlabel('People Displaced (Thousands)', fontsize=11)
ax.set_title('Top 15 Most Affected Districts — 2022 Pakistan Floods', fontsize=13, fontweight='bold')
ax.invert_yaxis()

# Legend for provinces
handles = [mpatches.Patch(color=c, label=p) for p, c in colors.items()
           if p in top15['province'].values]

import matplotlib.patches as mpatches
ax.legend(handles=handles, facecolor='#2f3436', labelcolor='white', fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('outputs/03_top_affected_districts.png', dpi=150, bbox_inches='tight', facecolor='#1a1d23')
plt.show()

## 6. Summary

In [ ]:
print("=" * 52)
print("  KEY FINDINGS — PAKISTAN FLOODS 2022")
print("=" * 52)
print(f"  Districts analysed       : {len(df)}")
print(f"  Total displaced          : {df['displaced'].sum():,}")
print(f"  Total deaths             : {df['deaths'].sum():,}")
print(f"  Total homes damaged      : {df['homes_damaged'].sum():,}")
print(f"  Avg % area flooded       : {df['affected_pct'].mean():.1f}%")
print()
print("  Vulnerability finding:")
print(f"  Districts with poverty >50% had {df[df['poverty_rate_pct']>50]['displaced_per_1000'].mean():.0f}")
print(f"  displaced/1000 vs {df[df['poverty_rate_pct']<30]['displaced_per_1000'].mean():.0f} in lower-poverty districts")
print()
print("  Most affected province by total displacement:")
print(f"  {df.groupby('province')['displaced'].sum().idxmax()}")